In [ ]:
import pandas as pd

files = [
    'airport_traffic_2019.csv',
    'airport_traffic_2020.csv',
    'airport_traffic_2021.csv',
    'airport_traffic_2022.csv'
]

dfs = [pd.read_csv(f) for f in files]
data = pd.concat(dfs, ignore_index=True)
data['FLT_DATE'] = pd.to_datetime(data['FLT_DATE'], format='%d-%m-%y')

print(data.shape)
data.head()

In [ ]:
data.info()
(data.isna().mean() * 100).sort_values(ascending=False)

In [ ]:
num_cols = ['FLT_DEP_1', 'FLT_ARR_1', 'FLT_TOT_1',
            'FLT_DEP_IFR_2', 'FLT_ARR_IFR_2', 'FLT_TOT_IFR_2']

data[num_cols].describe().T

In [ ]:
yearly = data.groupby('YEAR')[['FLT_DEP_1', 'FLT_ARR_1', 'FLT_TOT_1']].sum()
yearly['pct_change_tot'] = yearly['FLT_TOT_1'].pct_change() * 100
print(yearly)

In [ ]:
import matplotlib.pyplot as plt

yearly['FLT_TOT_1'].plot(kind='bar')
plt.title('Общий объем рейсов по годам')
plt.ylabel('Количество рейсов')
plt.xlabel('Год')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
monthly = data.groupby(['YEAR', 'MONTH_NUM'])['FLT_TOT_1'].sum().reset_index()
monthly_pivot = monthly.pivot(index='MONTH_NUM', columns='YEAR', values='FLT_TOT_1')
print(monthly_pivot)

monthly_pivot.plot(marker='o')
plt.title('Динамика общего количества рейсов по месяцам')
plt.xlabel('Месяц')
plt.ylabel('Количество рейсов')
plt.xticks(range(1, 13))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
daily_total = data.groupby('FLT_DATE')['FLT_TOT_1'].sum().reset_index()
daily_total['YEAR'] = daily_total['FLT_DATE'].dt.year
daily_total['MONTH'] = daily_total['FLT_DATE'].dt.month

daily_total.head()

In [ ]:
daily_total.groupby('YEAR')['FLT_TOT_1'].agg(['mean', 'median', 'std', 'min', 'max'])

In [ ]:
from scipy import stats

for year, grp in daily_total.groupby('YEAR'):
    stat, pvalue = stats.normaltest(grp['FLT_TOT_1'])
    print(year, stat, pvalue)

In [ ]:
groups = [grp['FLT_TOT_1'].values for _, grp in daily_total.groupby('YEAR')]
stats.kruskal(*groups)

In [ ]:
from itertools import combinations

years = sorted(daily_total['YEAR'].unique())

for y1, y2 in combinations(years, 2):
    x = daily_total[daily_total['YEAR'] == y1]['FLT_TOT_1']
    y = daily_total[daily_total['YEAR'] == y2]['FLT_TOT_1']
    stat, pvalue = stats.mannwhitneyu(x, y, alternative='two-sided')
    print(y1, y2, stat, pvalue)

In [ ]:
import numpy as np

daily_total['season'] = np.where(
    daily_total['MONTH'].isin([6, 7, 8]), 'summer',
    np.where(daily_total['MONTH'].isin([12, 1, 2]), 'winter', 'other')
)

summer = daily_total[daily_total['season'] == 'summer']['FLT_TOT_1']
winter = daily_total[daily_total['season'] == 'winter']['FLT_TOT_1']

stats.mannwhitneyu(summer, winter, alternative='two-sided')
print(summer.median(), winter.median())

In [ ]:
top_airports = (
    data.groupby(['APT_ICAO', 'APT_NAME'])['FLT_TOT_1']
    .sum()
    .sort_values(ascending=False)
    .head(15)
    .reset_index()
)

print(top_airports)

In [ ]:
plt.barh(top_airports['APT_NAME'], top_airports['FLT_TOT_1'])
plt.title('Топ-15 аэропортов по общему числу рейсов')
plt.xlabel('Количество рейсов')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
top_states = (
    data.groupby('STATE_NAME')['FLT_TOT_1']
    .sum()
    .sort_values(ascending=False)
    .head(15)
    .reset_index()
)

print(top_states)

In [ ]:
airport_year = data.groupby(['APT_ICAO', 'APT_NAME', 'YEAR'])['FLT_TOT_1'].sum().reset_index()
airport_pivot = airport_year.pivot_table(index=['APT_ICAO', 'APT_NAME'], columns='YEAR', values='FLT_TOT_1')

airport_pivot['recovery_2022_vs_2019'] = airport_pivot[2022] / airport_pivot[2019]
large_airports = airport_pivot[airport_pivot[2019] >= 100000].copy()

best_recovery = large_airports[['recovery_2022_vs_2019']].sort_values('recovery_2022_vs_2019', ascending=False).head(10)
worst_recovery = large_airports[['recovery_2022_vs_2019']].sort_values('recovery_2022_vs_2019').head(10)

print(best_recovery)
print(worst_recovery)

In [ ]:
ifr = data.dropna(subset=['FLT_TOT_IFR_2']).copy()
ifr = ifr[ifr['FLT_TOT_1'] > 0].copy()
ifr['ifr_share'] = ifr['FLT_TOT_IFR_2'] / ifr['FLT_TOT_1']

ifr['ifr_share'].describe()

In [ ]:
import pandas as pd

country_traffic = (
    data.groupby('STATE_NAME')['FLT_TOT_1']
    .sum()
    .reset_index()
)

country_traffic.head()

In [ ]:
import plotly.express as px

fig = px.choropleth(
    country_traffic,
    locations="STATE_NAME",
    locationmode="country names",
    color="FLT_TOT_1",
    color_continuous_scale="Viridis",
    title="Плотность авиаперелетов по странам (2019–2022)"
)

fig.show()

In [ ]:
import pandas as pd
import plotly.express as px

# 1. Берём только 2020 год
data_2020 = data[data['YEAR'] == 2020].copy()

# 2. На всякий случай создаём номер месяца, если его нет
data_2020['MONTH_NUM'] = data_2020['FLT_DATE'].dt.month

# 3. Группируем: страна + месяц
country_month_2020 = (
    data_2020
    .groupby(['STATE_NAME', 'MONTH_NUM'])['FLT_TOT_1']
    .sum()
    .reset_index()
)

# 4. Добавим красивые названия месяцев
month_names = {
    1: 'Январь', 2: 'Февраль', 3: 'Март', 4: 'Апрель',
    5: 'Май', 6: 'Июнь', 7: 'Июль', 8: 'Август',
    9: 'Сентябрь', 10: 'Октябрь', 11: 'Ноябрь', 12: 'Декабрь'
}

country_month_2020['MONTH_NAME'] = country_month_2020['MONTH_NUM'].map(month_names)

# 5. Строим анимированную карту по месяцам
fig = px.choropleth(
    country_month_2020,
    locations='STATE_NAME',
    locationmode='country names',
    color='FLT_TOT_1',
    animation_frame='MONTH_NAME',
    category_orders={'MONTH_NAME': list(month_names.values())},
    color_continuous_scale='Viridis',
    title='Плотность авиаперелётов по странам в 2020 году по месяцам',
    labels={
        'FLT_TOT_1': 'Количество рейсов',
        'STATE_NAME': 'Страна',
        'MONTH_NAME': 'Месяц'
    }
)

fig.update_layout(
    geo=dict(showframe=False, showcoastlines=True),
    title_x=0.5
)

fig.write_html("map_2020_by_month.html")
fig.show()